# Clase 8 — Introducción a MLOps

Una aplicación de movilidad estima la duración de un viaje antes de que comience. El modelo produce un número, pero el producto sólo funciona si también podemos responder:

- ¿con qué datos y código se construyó?;
- ¿qué versión está usando la aplicación?;
- ¿cómo sabemos si continúa funcionando cuando cambian los datos?;
- ¿quién puede detectar y corregir una falla?

MLOps organiza esas preguntas a lo largo del ciclo de vida de un sistema de machine learning.

## Antes de comenzar

Actualiza el repositorio público del curso desde su raíz y reconstruye el ambiente publicado por el profesor:

```bash
git status
git pull
uv sync --locked
```

`uv sync --locked` reconstruye el ambiente con las versiones registradas en `uv.lock`. En este repositorio no ejecutes `uv add` ni edites `pyproject.toml` o `uv.lock`.

**Prerrequisitos:** Python básico, lectura de tablas con pandas y nociones de modelo, artefacto y API.

**Resultado observable:** al terminar podrás explicar el ciclo completo de un sistema de ML, reconocer prácticas de MLOps y ubicar un proceso en los niveles de madurez 0–4.

## 1. Del modelo al sistema

Un modelo aprende una relación a partir de datos. Un **sistema de ML** conecta ese modelo con datos, código, una aplicación y personas que toman decisiones.

En el caso de NYC Taxi, la predicción pasa por varias piezas: registros de viajes, reglas de preparación, un modelo entrenado, una API y una aplicación que muestra la duración estimada. Después del viaje llega información nueva que permite revisar el comportamiento del sistema.

![El modelo de duración conectado con datos, preparación, API, usuario y observación](../assets/modulo-02-ciclo-mlops/clase-08/sistema-ml.svg)

**Pregunta inicial:** si la API responde correctamente pero la duración estimada es poco útil, ¿qué partes del sistema revisarías antes de volver a entrenar?

## 2. ¿Qué es MLOps?

**MLOps** (*Machine Learning Operations*) es una práctica de ingeniería y colaboración que une el desarrollo de sistemas de ML con su operación. Busca que los datos, el código, los modelos y la aplicación puedan integrarse, probarse, desplegarse, observarse y mejorarse de manera confiable.

No es una herramienta específica. Es una forma de organizar responsabilidades y prácticas durante todo el ciclo de vida:

1. definir el problema y los criterios de éxito;
2. preparar datos y desarrollar candidatos;
3. evaluar y decidir qué versión puede usarse;
4. integrar el modelo con una aplicación;
5. observar el sistema y usar esa información en el siguiente ciclo.

MLOps se apoya en ideas de **DevOps**, donde desarrollo y operación colaboran para entregar software con cambios pequeños, verificables y frecuentes. En ML, además del código, cambian los datos y los modelos. La guía [MLOps: Continuous delivery and automation pipelines in machine learning](https://docs.cloud.google.com/architecture/mlops-continuous-delivery-and-automation-pipelines-in-machine-learning), de Google Cloud, muestra cómo CI, CD y CT se adaptan a estos sistemas.

### CI, CD y CT en sistemas de ML

Un **pipeline** es una secuencia definida de pasos con entradas y salidas: por ejemplo, preparar datos → entrenar → evaluar → producir un artefacto.

| Práctica | Pregunta que responde | En un sistema de ML |
|---|---|---|
| **CI — integración continua** | ¿el cambio se integra sin romper lo existente? | prueba código, preparación de datos, esquemas y comportamiento del modelo |
| **CD — entrega o despliegue continuo** | ¿una versión validada puede llegar de forma confiable al entorno objetivo? | prepara y promueve código, pipeline y artefactos del modelo |
| **CT — entrenamiento continuo** | ¿cuándo y cómo se produce un nuevo candidato? | ejecuta de nuevo preparación, entrenamiento y evaluación ante datos o eventos definidos |

**Continuo** no significa “sin control humano”. Significa que el proceso está definido, deja evidencia y puede repetirse. Una organización puede automatizar algunos pasos y mantener aprobaciones manuales donde el riesgo lo justifique.

## 3. 🧩 Componentes principales de MLOps

Un sistema de ML en producción es más que un modelo entrenado. MLOps coordina cuatro componentes para que funcionen como una unidad reproducible y mantenible.

### 3.1 Datos

- **Ingesta:** obtener los datos desde archivos, bases, APIs o flujos de eventos.
- **Validación:** comprobar esquema, tipos, rangos, valores faltantes y reglas del dominio.
- **Preparación:** convertir los datos crudos en las features y el target que necesita el entrenamiento.
- **Versionado:** identificar la fuente y el corte exacto utilizados por cada ejecución.

En NYC Taxi debemos distinguir, por ejemplo, el mes de los viajes, las reglas para calcular duración y las filas descartadas antes de entrenar.

### 3.2 Modelos

- **Entrenamiento:** ejecutar un algoritmo con datos y configuración definidos.
- **Evaluación:** medir el candidato con datos separados y criterios comparables.
- **Empaquetado:** guardar el modelo junto con la preparación que necesita para predecir.
- **Gestión:** identificar versiones, estado y relación con sus métricas.

Un **artefacto** es el objeto guardado que una aplicación puede cargar para predecir; suele acompañarse de metadatos como features, versión y métrica.

### 3.3 Código

- **Componentes reutilizables:** separar preparación, entrenamiento, evaluación e inferencia.
- **Pruebas:** comprobar funciones, integración entre piezas y contrato de la API.
- **Ambiente reproducible:** declarar Python y dependencias necesarias.
- **Entrega:** preparar el código y los artefactos para el entorno donde se usarán.

El mismo procesamiento debe aplicarse al entrenar y al predecir. Si la API calcula las features de otra manera, el modelo recibe entradas distintas de aquellas con las que aprendió.

### 3.4 Procesos y personas

- **Criterios de aprobación:** decidir cuándo un candidato puede avanzar.
- **Despliegue y recuperación:** activar una versión y poder regresar a otra.
- **Monitoreo:** observar datos, servicio y desempeño del modelo.
- **Responsabilidades:** acordar quién revisa, aprueba y responde ante una falla.

La automatización ayuda cuando las entradas, salidas, criterios y responsabilidades del proceso están claras. Consulta el diagrama de [Google Cloud sobre los elementos que rodean al código de ML](https://docs.cloud.google.com/architecture/mlops-continuous-delivery-and-automation-pipelines-in-machine-learning) para comparar estos componentes con una arquitectura de producción.

## 4. ⚠️ Retos que MLOps ayuda a abordar

La sección anterior identificó las piezas del sistema. Ahora veremos qué ocurre cuando esas piezas no están coordinadas.

### 4.1 Reproducibilidad y gestión de versiones

- **Problema:** es difícil reconstruir un resultado si desconocemos datos, código, configuración o ambiente.
- **Riesgo:** dos archivos con el mismo nombre pueden representar modelos distintos y no sabemos cuál usa la aplicación.
- **Respuesta MLOps:** asociar cada ejecución con sus entradas, métrica, artefacto y versión.

### 4.2 Integración entre personas y sistemas

- **Problema:** el modelo pasa del notebook a una API y cada parte puede aplicar reglas diferentes.
- **Riesgo:** otra persona debe reescribir el procesamiento o depender de instrucciones incompletas.
- **Respuesta MLOps:** interfaces claras, código reutilizable, ambiente definido y responsabilidades compartidas.

### 4.3 Pipelines frágiles y deuda técnica

- **Problema:** datos, features y código tienen dependencias que pueden permanecer ocultas.
- **Riesgo:** cambiar una columna rompe otra etapa o altera el resultado sin una señal clara.
- **Respuesta MLOps:** modularizar, validar entradas y probar la conexión entre etapas.

La **deuda técnica** es el costo futuro de mantener decisiones que resolvieron algo rápido, pero dejaron el sistema difícil de cambiar o explicar. El artículo [Hidden Technical Debt in Machine Learning Systems](https://research.google/pubs/hidden-technical-debt-in-machine-learning-systems/) desarrolla este problema con ejemplos de sistemas reales.

### 4.4 Escalabilidad y rendimiento

- **Problema:** un proceso que funciona con una muestra local puede tardar demasiado o fallar con más solicitudes y datos.
- **Riesgo:** entrenamiento lento, predicciones tardías o uso excesivo de recursos.
- **Respuesta MLOps:** medir tiempos y recursos, separar cargas y elegir una forma de ejecución acorde con el uso.

### 4.5 Monitoreo y cambios en los datos

- **Problema:** después del despliegue pueden cambiar los datos, el comportamiento de usuarios o la relación que aprendió el modelo.
- **Riesgo:** la API continúa respondiendo aunque las predicciones pierdan utilidad.
- **Respuesta MLOps:** observar calidad de datos, errores, latencia y desempeño cuando la respuesta real esté disponible.

Llamamos **data drift** a un cambio en la distribución de las entradas. Detectarlo es una señal para investigar; no demuestra por sí solo que el modelo deba reentrenarse.

## 5. 🛠️ Mejores prácticas iniciales en MLOps

Los retos explican **qué puede fallar**. Las prácticas siguientes describen **cómo reducir esos riesgos** de manera incremental. La [guía de Google Cloud sobre CI, CD y CT para ML](https://docs.cloud.google.com/architecture/mlops-continuous-delivery-and-automation-pipelines-in-machine-learning) ofrece arquitecturas de referencia para profundizar.

### 5.1 Controlar versiones de los elementos relevantes

- Identificar el código que produjo una ejecución.
- Registrar la fuente y el corte de datos utilizados.
- Conservar configuración, métricas y versión del artefacto.
- Definir cómo recuperar una versión anterior.

### 5.2 Automatizar pruebas y validaciones

- **Datos:** revisar columnas, tipos, rangos y valores faltantes.
- **Modelo:** comprobar que la métrica y otros criterios cumplen lo acordado.
- **Integración:** verificar que el artefacto puede cargarse y que la API respeta el contrato.

Una validación útil produce evidencia y detiene el proceso cuando no se cumple una condición importante.

### 5.3 Aplicar CI, CD y CT según la necesidad

- Integrar cambios pequeños y ejecutar pruebas automáticamente.
- Preparar de forma repetible una versión candidata.
- Automatizar entrenamiento o despliegue sólo cuando existan criterios claros.
- Mantener aprobaciones humanas cuando el riesgo o el contexto lo requieran.

### 5.4 Mantener paridad entre desarrollo y uso

- Reutilizar la misma definición de features al entrenar y al predecir.
- Reconstruir el ambiente a partir de dependencias declaradas.
- Probar el modelo con entradas equivalentes a las que recibirá la aplicación.

La diferencia involuntaria entre el procesamiento de entrenamiento y el de inferencia se conoce como **training-serving skew**.

### 5.5 Registrar y comparar ejecuciones

- Asociar parámetros, datos, métricas y artefactos.
- Comparar candidatos bajo condiciones conocidas.
- Documentar por qué una versión fue promovida o rechazada.

### 5.6 Monitorear y definir alertas útiles

- Medir disponibilidad, errores y latencia del servicio.
- Revisar calidad y cambios de los datos de entrada.
- Evaluar el modelo cuando llegue la respuesta real.
- Asignar a cada alerta una acción y una persona responsable.

### 5.7 Construir cultura y documentación

- Compartir supuestos, decisiones y criterios.
- Revisar cambios entre disciplinas, no sólo dentro de ciencia de datos.
- Diseñar procedimientos que otra persona pueda ejecutar y auditar.
- Incorporar lo aprendido en operación al siguiente ciclo de diseño.

### 5.8 Beneficios observables

- **Velocidad:** repetir y revisar el proceso toma menos tiempo.
- **Confiabilidad:** las pruebas y criterios reducen cambios defectuosos.
- **Escalabilidad:** un proceso definido puede atender más datos, modelos o personas sin depender de pasos improvisados.
- **Trazabilidad:** es posible explicar qué versión produjo una salida y por qué fue aprobada.
- **Gobierno:** existen reglas, responsabilidades y evidencia para tomar decisiones sobre el sistema.

Estas prácticas son acumulativas: versionar sin probar deja riesgos abiertos; monitorear sin responsables produce alertas que nadie atiende.

## 6. 🕰️ Evolución histórica del enfoque MLOps

MLOps se desarrolló a partir de la necesidad de operar modelos con el mismo rigor que otros sistemas de software, incorporando además los cambios propios de datos y modelos.

### 6.1 Antes de 2015: procesos artesanales

- Entrenamiento interactivo en notebooks o scripts locales.
- Modelos compartidos como archivos sin suficiente contexto.
- Pruebas, despliegues y monitoreo principalmente manuales.
- Dependencia fuerte del conocimiento de una persona.

### 6.2 2015: la deuda técnica se vuelve visible

- En 2015, Google Research publicó [*Hidden Technical Debt in Machine Learning Systems*](https://research.google/pubs/hidden-technical-debt-in-machine-learning-systems/), donde documentó dependencias de datos, configuración, infraestructura y comportamiento externo.
- El código que implementa el modelo apareció como una parte pequeña del sistema completo.
- La mantenibilidad comenzó a discutirse junto con la precisión predictiva.

### 6.3 2016–2019: pipelines y plataformas integradas

- Se consolidaron herramientas para orquestar preparación, entrenamiento y evaluación.
- Los servicios administrados comenzaron a conectar datos, cómputo, modelos y despliegue.
- Las ideas de CI y CD se adaptaron para validar también datos y modelos; la guía de [Google Cloud sobre pipelines MLOps](https://docs.cloud.google.com/architecture/mlops-continuous-delivery-and-automation-pipelines-in-machine-learning) resume esta evolución.
- El entrenamiento continuo agregó la posibilidad de producir candidatos mediante eventos o calendarios definidos.

### 6.4 Desde 2020: operación, gobierno y nuevos tipos de modelos

- Los [modelos de madurez de MLOps](https://learn.microsoft.com/es-es/azure/architecture/ai-ml/guide/mlops-maturity-model) ayudaron a diagnosticar capacidades y planear mejoras incrementales.
- Monitoreo, trazabilidad y gobierno se integraron al ciclo de vida.
- La IA responsable amplió las revisiones de calidad con sesgo, explicabilidad y riesgo.
- MLOps se extendió a dispositivos y sistemas generativos. Microsoft explica que [GenAIOps complementa las capacidades de MLOps](https://learn.microsoft.com/en-us/azure/well-architected/ai/mlops-genaiops) con prácticas propias de modelos generativos.

![Línea de tiempo desde procesos manuales hasta operación observable](../assets/modulo-02-ciclo-mlops/clase-08/evolucion-mlops.svg)

La evolución no consiste en acumular herramientas: consiste en coordinar mejor el ciclo completo y hacer visibles las condiciones de cada decisión.

## 7. 🔄 Un ciclo, no una línea de llegada

El trabajo se repite en tres ciclos relacionados:

- **Diseño de la solución:** define usuario, decisión, datos disponibles y criterios de éxito.
- **Desarrollo del modelo:** prepara datos, experimenta, evalúa y empaqueta un candidato.
- **Operación:** integra una versión, observa su uso y produce información para el siguiente ciclo.

![Proceso iterativo de MLOps con ciclos de diseño, desarrollo del modelo y operación](../assets/modulo-02-ciclo-mlops/clase-08/ciclo-incremental-innoq.png)

*Fuente: [INNOQ — MLOps Principles](https://ml-ops.org/content/mlops-principles), CC BY 4.0.*

Las flechas de regreso son tan importantes como las de avance: una falla en operación puede revelar un problema de datos, una transformación inconsistente o un criterio de diseño incompleto.

### Video — reconocer el ciclo en otro ejemplo

Observa el fragmento **00:32–02:51** de [MLOps Zoomcamp 1.1 — Introduction](https://www.youtube.com/watch?v=s0uaFZSzwfI), de DataTalksClub.

Mientras lo ves, anota:

1. ¿qué elementos existen además del modelo?;
2. ¿qué pasos necesitan repetirse?;
3. ¿qué información regresa desde el uso del sistema?

Después compara tus respuestas con el ciclo anterior.

## 8. 🚕 Caso NYC Taxi: del entrenamiento a una versión utilizable

Retomaremos la estimación de duración de viajes con las mismas variables preparadas este semestre:

Las **features** son las entradas disponibles al solicitar la predicción; el **target** es la duración real que el modelo intenta estimar y que conocemos en viajes ya terminados.

```python
FEATURES = [
    "distancia_km",
    "pasajeros",
    "hora_recoleccion",
    "zona_origen",
    "zona_destino",
]
TARGET = "duracion_minutos"
```

La muestra local contiene 24 viajes terminados. Usaremos 18 para ajustar un **baseline constante** —siempre predice la media calculada con los datos de ajuste— y seis para comprobar el recorrido sobre filas no utilizadas en el ajuste. Su sencillez permite inspeccionar con claridad datos, configuración, métrica, artefacto y versión.

La siguiente celda localiza la raíz mediante `pyproject.toml`, lee el CSV y aplica los mismos nombres de zonas que utiliza el código de preparación del semestre.

In [5]:
from pathlib import Path

import pandas as pd

FEATURES = [
    "distancia_km",
    "pasajeros",
    "hora_recoleccion",
    "zona_origen",
    "zona_destino",
]
TARGET = "duracion_minutos"

candidate_roots = [Path.cwd(), *Path.cwd().parents]
repo_root = next(
    (root for root in candidate_roots if (root / "pyproject.toml").is_file()),
    None,
)
if repo_root is None:
    raise FileNotFoundError("Ejecuta el notebook dentro del repositorio del curso")

data_path = (
    repo_root
    / "labs/starters/clase-06-api-prediccion"
    / "muestra-green-taxi-2026-03.csv"
)
trips = (
    pd.read_csv(data_path)
    .rename(
        columns={
            "PULocationID": "zona_origen",
            "DOLocationID": "zona_destino",
        }
    )
    [FEATURES + [TARGET]]
)

trips.head()

,distancia_km,pasajeros,hora_recoleccion,zona_origen,zona_destino,duracion_minutos
0,10.78,1,4,255,246,29.350000
1,4.92,1,15,74,244,23.800000
2,1.45,1,7,74,74,6.983333
3,3.60,1,17,74,236,12.916667
4,8.88,1,1,157,148,13.750000


### Ajustar y evaluar

El **RMSE** resume el tamaño típico del error en las mismas unidades del target; aquí se interpreta en minutos. Primero calculamos la diferencia entre cada duración real y la predicción, elevamos esas diferencias al cuadrado, promediamos y obtenemos la raíz cuadrada.

La partición es fija: las primeras 18 filas se usan para ajuste y las últimas seis para validación. Así, otra persona puede repetir exactamente la misma demostración.

In [6]:
from math import sqrt

train_data = trips.iloc[:18].copy()
validation_data = trips.iloc[18:].copy()

baseline_mean = train_data[TARGET].mean()
predictions = pd.Series(baseline_mean, index=validation_data.index)
validation_rmse = sqrt(((validation_data[TARGET] - predictions) ** 2).mean())

pd.DataFrame(
    {
        "training_rows": [len(train_data)],
        "validation_rows": [len(validation_data)],
        "baseline_mean_min": [round(baseline_mean, 2)],
        "validation_rmse_min": [round(validation_rmse, 2)],
    }
)

,training_rows,validation_rows,baseline_mean_min,validation_rmse_min
0,18,6,13.55,19.09


**Resultado esperado:** 18 filas de entrenamiento, seis de validación, `baseline_mean` de **13.55 minutos** y RMSE de **19.09 minutos**.

El RMSE alto muestra que predecir siempre el promedio es insuficiente para viajes muy distintos. La muestra pequeña sirve para verificar el proceso y comparar candidatos bajo las mismas condiciones; no permite estimar el desempeño para toda la ciudad.

El recorrido ya contiene elementos del código usado este semestre: definición compartida de features y target, separación de entrenamiento y validación, una métrica y un resultado que después consumiría la API.

### Conservar las condiciones de cada ejecución

En el entrenamiento del semestre, el artefacto reúne el modelo, el orden de las features, una versión y el RMSE de validación. Adaptaremos la misma estructura y agregaremos un registro pequeño con los datos y la configuración que produjeron cada candidato.

In [7]:
def run_training(
    data: pd.DataFrame,
    duration_limit: int,
    model_version: str,
) -> tuple[dict, dict]:
    train_data = data.iloc[:18].copy()
    validation_data = data.iloc[18:].copy()
    train_data = train_data.loc[train_data[TARGET] <= duration_limit]

    baseline_mean = train_data[TARGET].mean()
    predictions = pd.Series(baseline_mean, index=validation_data.index)
    validation_rmse = sqrt(
        ((validation_data[TARGET] - predictions) ** 2).mean()
    )

    model_artifact = {
        "model": {"type": "mean_baseline", "value": float(baseline_mean)},
        "features": FEATURES,
        "version": model_version,
        "validation_rmse": float(validation_rmse),
    }
    run_record = {
        "model_version": model_version,
        "dataset": data_path.name,
        "training_rows": len(train_data),
        "config": {"duration_limit": duration_limit},
        "validation_rmse": round(validation_rmse, 2),
    }
    return model_artifact, run_record


artifact_v1, run_v1 = run_training(
    trips,
    duration_limit=60,
    model_version="green-taxi-mean-baseline-v1",
)
artifact_v2, run_v2 = run_training(
    trips,
    duration_limit=15,
    model_version="green-taxi-mean-baseline-v2",
)

pd.DataFrame([run_v1, run_v2])

,model_version,dataset,training_rows,config,validation_rmse
0,green-taxi-mean-baseline-v1,muestra-green-taxi-2026-03.csv,18,{'duration_limit': 60},19.09
1,green-taxi-mean-baseline-v2,muestra-green-taxi-2026-03.csv,14,{'duration_limit': 15},19.86


Las dos ejecuciones usan el mismo código y la misma validación, pero cambia una regla de preparación. El registro permite explicar por qué el segundo candidato aprendió con menos filas y comparar sus métricas sin depender de la memoria de quien ejecutó el notebook.

| Versión | Límite | Filas para ajuste | RMSE esperado |
|---|---:|---:|---:|
| `green-taxi-mean-baseline-v1` | 60 min | 18 | 19.09 min |
| `green-taxi-mean-baseline-v2` | 15 min | 14 | 19.86 min |

Registrar una ejecución no decide automáticamente qué modelo usar. **Promover** un candidato significa aprobarlo para sustituir la versión activa; para hacerlo necesitamos un criterio explícito.

In [8]:
rmse_threshold = 19.50
active_artifact = artifact_v1
candidate_artifact = artifact_v2

if candidate_artifact["validation_rmse"] <= rmse_threshold:
    active_artifact = candidate_artifact
    promotion_decision = "candidato promovido"
else:
    promotion_decision = "candidato rechazado; se conserva la versión activa"

{
    "promotion_decision": promotion_decision,
    "active_version": active_artifact["version"],
    "active_rmse": round(active_artifact["validation_rmse"], 2),
}

{'promotion_decision': 'candidato rechazado; se conserva la versión activa',
 'active_version': 'green-taxi-mean-baseline-v1',
 'active_rmse': 19.09}

### Usar el artefacto desde una aplicación

La API del semestre carga un artefacto, construye una entrada con el orden de features esperado y devuelve tanto la estimación como la versión del modelo. La misma idea puede observarse con una función local:

In [9]:
def predict(request: dict, model_artifact: dict) -> dict:
    missing_features = [
        feature
        for feature in model_artifact["features"]
        if feature not in request
    ]
    if missing_features:
        raise ValueError(f"Faltan features: {missing_features}")

    return {
        "duracion_estimada_minutos": round(model_artifact["model"]["value"], 1),
        "version_modelo": model_artifact["version"],
    }


sample_request = trips.loc[20, FEATURES].to_dict()
predict(sample_request, active_artifact)

{'duracion_estimada_minutos': 13.6,
 'version_modelo': 'green-taxi-mean-baseline-v1'}

### El ciclo completo del caso

![Flujo desde datos de viajes hasta una API y la observación del uso](../assets/modulo-02-ciclo-mlops/clase-08/flujo-nyc-taxi.svg)

En la demostración ya identificamos datos, preparación, entrenamiento, evaluación, artefacto y una forma de inferencia. Para operar el sistema todavía faltarían, entre otras capacidades:

1. ejecutar el pipeline fuera del notebook;
2. conservar registros y artefactos en un lugar compartido;
3. probar automáticamente datos, código y compatibilidad con la API;
4. desplegar una versión aprobada y poder regresar a la anterior;
5. observar errores, latencia y desempeño cuando llegue la duración real.

## 9. 📈 Niveles de madurez en MLOps

El [modelo de madurez de MLOps de Microsoft](https://learn.microsoft.com/es-es/azure/architecture/ai-ml/guide/mlops-maturity-model) ayuda a describir las capacidades actuales, encontrar brechas y elegir el siguiente incremento. No califica qué tan “bueno” es un equipo ni obliga a alcanzar el nivel máximo.

El modelo 0–4 considera tres dimensiones:

- **personas y cultura:** quién colabora y quién puede responder ante una falla;
- **procesos:** qué pasos son manuales, repetibles, aprobados o automáticos;
- **tecnología:** cómo se prueban, registran, despliegan y observan datos, código y modelos.

![Escalera con los cinco niveles de madurez MLOps](../assets/modulo-02-ciclo-mlops/clase-08/madurez-mlops.svg)

### Nivel 0 — Sin MLOps

- **Descripción:**
  - El ciclo completo del modelo es difícil de gestionar.
  - Los equipos trabajan de forma separada y reciben poca retroalimentación después del despliegue.
- **Aspectos destacados:**
  - Entrenamiento, pruebas y despliegue manuales.
  - Versiones difíciles de rastrear.
  - Sin seguimiento centralizado del desempeño.
  - Adecuado para una prueba de concepto que cambia poco.

### Nivel 1 — DevOps, pero sin MLOps

- **Descripción:**
  - Publicar la aplicación es más confiable, pero cada modelo nuevo todavía depende de un proceso manual.
  - El comportamiento del modelo en producción sigue siendo difícil de rastrear y reproducir.
- **Aspectos destacados:**
  - Código de la aplicación bajo control de versiones.
  - Construcción, pruebas y despliegue de la aplicación automatizados.
  - Métricas operativas básicas del servicio.
  - Entrenamiento y registro de experimentos todavía manuales.

### Nivel 2 — Entrenamiento automatizado

- **Descripción:**
  - El ambiente de entrenamiento es gestionado y las ejecuciones son trazables.
  - El modelo puede reproducirse con facilidad; su despliegue todavía requiere una decisión manual.
- **Aspectos destacados:**
  - Pipeline automatizado de preparación, entrenamiento y evaluación.
  - Parámetros, métricas y artefactos asociados a cada ejecución.
  - Código y modelos versionados o identificados.
  - Colaboración directa entre ciencia e ingeniería de datos.

### Nivel 3 — Despliegue automático del modelo

- **Descripción:**
  - Los candidatos aprobados pueden desplegarse automáticamente.
  - La versión activa puede rastrearse hasta sus datos, código y configuración.
- **Aspectos destacados:**
  - Pruebas automáticas para código, datos, modelo e integración.
  - Criterios explícitos de promoción.
  - Despliegue coordinado del modelo y la aplicación.
  - Procedimiento verificable para regresar a una versión anterior.

### Nivel 4 — Operación automatizada

- **Descripción:**
  - El sistema completo es observable y los datos de producción alimentan la mejora.
  - Señales acordadas pueden iniciar un nuevo ciclo controlado de entrenamiento, evaluación y promoción.
- **Aspectos destacados:**
  - Entrenamiento y pruebas automatizados.
  - Métricas centralizadas del servicio, los datos y el modelo desplegado.
  - Monitoreo de calidad, cambios y frescura de datos.
  - Promoción basada en políticas, con alertas y responsables definidos.

### El mismo caso a través de los cinco niveles

| Nivel | Aplicación de duración de viajes |
|---:|---|
| 0 | una persona ejecuta el notebook, guarda `modelo.pkl` y lo entrega manualmente |
| 1 | la API tiene pruebas y despliegue reproducible, pero recibe modelos creados manualmente |
| 2 | un pipeline prepara datos, entrena, evalúa y registra cada candidato; alguien decide el despliegue |
| 3 | un candidato que supera las pruebas se promueve automáticamente y conserva trazabilidad completa |
| 4 | el sistema observa datos, errores y desempeño; las señales acordadas alimentan un nuevo ciclo controlado |

La función `run_training` reúne algunos elementos de un pipeline, pero sigue ejecutándose manualmente dentro de un notebook. Por eso la demostración no implementa por sí sola un nivel de madurez completo.

## 10. 🧠 Práctica — diagnosticar la madurez

Trabajen en parejas. Desde la raíz del repositorio confirmen primero su ubicación y creen la carpeta local:

```bash
git status
mkdir -p labs/trabajo-local/clase-08
```

En VS Code creen `labs/trabajo-local/clase-08/diagnostico-madurez.md`. Analicen estos cinco escenarios de la aplicación NYC Taxi:

1. El modelo se entrena en un notebook y se envía por mensaje a quien mantiene la API.
2. La API tiene pruebas automáticas y se despliega al integrar cambios; el modelo todavía se reemplaza a mano.
3. Un job ejecuta preparación, entrenamiento y evaluación, y conserva el registro de cada candidato; una persona aprueba el despliegue.
4. Las pruebas promueven automáticamente una versión y permiten rastrearla hasta datos, código y configuración.
5. El servicio registra señales de datos y desempeño; una regla aprobada inicia un pipeline que evalúa un nuevo candidato.

Para cada escenario escriban:

- nivel de madurez y una evidencia que lo justifica;
- principal riesgo que todavía permanece;
- siguiente capacidad que agregarían;
- dimensión principal del cambio: personas, proceso o tecnología.

✅ **Checkpoint:** intercambien el archivo con otra pareja. Si asignaron niveles distintos, comparen la evidencia; el número sin justificación no es suficiente.

## Reto opcional

Diseña dos comprobaciones para el paso entre los niveles 2 y 3 del caso NYC Taxi:

1. una prueba que pueda bloquear la promoción de un candidato;
2. una verificación posterior al despliegue que confirme qué versión está activa.

Para cada una indica entrada, condición de éxito, evidencia producida y quién debe responder si falla.

## Errores frecuentes

- **“MLOps es poner el modelo en una nube”.** La infraestructura puede ayudar, pero MLOps también incluye datos, pruebas, versiones, personas y monitoreo.
- **“Más automatización siempre significa más madurez”.** Automatizar sin criterios ni evidencia puede amplificar errores.
- **“HTTP 200 demuestra que el modelo funciona bien”.** Sólo confirma que la solicitud fue procesada; no mide la utilidad de la predicción.
- **“Git versiona automáticamente los datos y modelos”.** Git identifica bien código y texto pequeño; otros artefactos necesitan una estrategia explícita.
- **“Un reentrenamiento siempre mejora el modelo”.** Cada candidato debe evaluarse bajo criterios comparables antes de sustituir la versión activa.

## Recopilación

- Un modelo es sólo una parte de un sistema de ML.
- MLOps conecta desarrollo y operación mediante colaboración, automatización, pruebas, trazabilidad y observación.
- Datos, modelo, código y procesos deben poder evolucionar juntos.
- En NYC Taxi, una ejecución identificable asocia datos, configuración, métrica y artefacto; un criterio decide si el candidato puede sustituir la versión activa.
- Los niveles 0–4 describen capacidades crecientes: proceso manual, DevOps de la aplicación, entrenamiento automatizado, despliegue automático y operación automatizada.
- El siguiente paso no es adoptar todas las herramientas, sino identificar la brecha que más riesgo reduce.

En la siguiente clase profundizaremos en cómo diagnosticar esas capacidades y convertir una brecha en una ruta incremental de mejora.

## Check final de la clase

Responde sin consultar las secciones anteriores:

1. ¿Qué agrega MLOps al desarrollo de un modelo?
2. ¿Por qué CI en ML debe revisar algo más que código?
3. ¿Qué cuatro elementos permiten identificar una ejecución reproducible?
4. ¿Cuál es la diferencia principal entre los niveles 2 y 3?
5. En el caso NYC Taxi, ¿qué información necesitarías para regresar a la versión anterior?

La clase está completa cuando puedes justificar tus respuestas con un ejemplo del ciclo NYC Taxi y tu archivo `diagnostico-madurez.md` distingue evidencia, riesgo y siguiente capacidad.

## Referencias

- [Google Cloud — MLOps: Continuous delivery and automation pipelines in machine learning](https://docs.cloud.google.com/architecture/mlops-continuous-delivery-and-automation-pipelines-in-machine-learning)
- [Microsoft Learn — Modelo de madurez de MLOps](https://learn.microsoft.com/es-es/azure/architecture/ai-ml/guide/mlops-maturity-model)
- [Google Research — Hidden Technical Debt in Machine Learning Systems](https://research.google/pubs/hidden-technical-debt-in-machine-learning-systems/)
- [Microsoft Learn — MLOps and GenAIOps for AI workloads](https://learn.microsoft.com/en-us/azure/well-architected/ai/mlops-genaiops)
- [INNOQ — MLOps Principles](https://ml-ops.org/content/mlops-principles)
- [DataTalksClub — MLOps Zoomcamp 1.1: Introduction](https://www.youtube.com/watch?v=s0uaFZSzwfI)

Fuentes consultadas el 9 de septiembre de 2026.